<a href="https://colab.research.google.com/github/SBethune103/virtual-running-coach-pipeline/blob/dev/notebooks/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q polars pyarrow

import polars as pl
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/virtual-running-coach")
DATA_RAW = BASE / "data/raw"
DATA_PROCESSED = BASE / "data/processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")

Mounted at /content/drive
✅ Setup complete


In [ ]:
# Find all weekly files
weekly_files = sorted(list(DATA_RAW.glob("run_ww_*_w.csv")))
print(f"Found {len(weekly_files)} weekly files")

# Load and combine (limit to first few if space is tight)
dfs = []
for f in weekly_files[:4]:  # Start with 4 files to stay light
    print(f"Loading {f.name}...")
    df = pl.read_csv(f, try_parse_dates=True, null_values=["", "NA"])
    dfs.append(df)

df_all = pl.concat(dfs)
print(f"\nCombined shape: {df_all.shape[0]:,} rows × {df_all.shape[1]} columns")

Found 2 weekly files
Loading run_ww_2019_w.csv...
Loading run_ww_2020_w.csv...

Combined shape: 3,786,848 rows × 9 columns


In [ ]:
def clean_and_feature(df: pl.DataFrame) -> pl.DataFrame:
    # Handle datetime column
    if df["datetime"].dtype == pl.Utf8:
        df = df.with_columns(pl.col("datetime").str.to_datetime(strict=False).alias("date"))
    else:
        df = df.with_columns(pl.col("datetime").alias("date"))

    df = (
        df
        .filter(pl.col("distance") > 0)
        .with_columns([
            (pl.col("distance") / (pl.col("duration") / 60)).alias("avg_speed_kmh"),
            (pl.col("duration") / pl.col("distance")).alias("pace_min_per_km"),
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month"),
            pl.col("date").dt.weekday().alias("weekday"),
        ])
        .drop_nulls(subset=["distance", "duration"])
    )
    return df

df_clean = clean_and_feature(df_all)
print(f"After cleaning: {df_clean.shape[0]:,} rows")

After cleaning: 2,756,863 rows


In [ ]:
# Create athlete summaries
athlete_summary = (
    df_clean
    .group_by("athlete")
    .agg([
        pl.col("distance").sum().alias("total_distance_km"),
        pl.col("distance").mean().alias("avg_weekly_distance"),
        pl.col("distance").count().alias("weeks_active"),
        pl.col("pace_min_per_km").mean().alias("avg_pace"),
        pl.col("avg_speed_kmh").mean().alias("avg_speed"),
        pl.col("gender").first().alias("gender"),
        pl.col("age_group").first().alias("age_group"),
        pl.col("country").first().alias("country"),
    ])
    .with_columns([
        (pl.col("total_distance_km") / pl.col("weeks_active")).alias("consistency_score")
    ])
)

print(f"Athletes summarised: {athlete_summary.shape[0]:,}")
print(athlete_summary.head())

Athletes summarised: 36,412
shape: (5, 10)
┌─────────┬────────────┬────────────┬────────────┬───┬────────┬───────────┬────────────┬───────────┐
│ athlete ┆ total_dist ┆ avg_weekly ┆ weeks_acti ┆ … ┆ gender ┆ age_group ┆ country    ┆ consisten │
│ ---     ┆ ance_km    ┆ _distance  ┆ ve         ┆   ┆ ---    ┆ ---       ┆ ---        ┆ cy_score  │
│ i64     ┆ ---        ┆ ---        ┆ ---        ┆   ┆ str    ┆ str       ┆ str        ┆ ---       │
│         ┆ f64        ┆ f64        ┆ u32        ┆   ┆        ┆           ┆            ┆ f64       │
╞═════════╪════════════╪════════════╪════════════╪═══╪════════╪═══════════╪════════════╪═══════════╡
│ 31697   ┆ 1972.555   ┆ 31.310397  ┆ 63         ┆ … ┆ M      ┆ 18 - 34   ┆ United     ┆ 31.310397 │
│         ┆            ┆            ┆            ┆   ┆        ┆           ┆ Kingdom    ┆           │
│ 9633    ┆ 835.3025   ┆ 16.37848   ┆ 51         ┆ … ┆ F      ┆ 18 - 34   ┆ United     ┆ 16.37848  │
│         ┆            ┆            ┆           

In [ ]:
# Save only what we need
df_clean.write_parquet(DATA_PROCESSED / "running_combined_clean.parquet")
athlete_summary.write_parquet(DATA_PROCESSED / "athlete_summary.parquet")

print("✅ Files saved:")
print(" - running_combined_clean.parquet")
print(" - athlete_summary.parquet")

✅ Files saved:
 - running_combined_clean.parquet
 - athlete_summary.parquet


In [ ]:
print("=== Key Insights ===")
print(f"Total athletes: {athlete_summary.shape[0]:,}")
print(f"Average weeks active: {athlete_summary['weeks_active'].mean():.1f}")
print(f"Median weekly distance: {athlete_summary['avg_weekly_distance'].median():.1f} km")

print("\nTop countries by number of athletes:")
print(
    athlete_summary
    .group_by("country")
    .agg(pl.len().alias("athletes"))
    .sort("athletes", descending=True)
    .head(8)
)

=== Key Insights ===
Total athletes: 36,412
Average weeks active: 75.7
Median weekly distance: 29.1 km

Top countries by number of athletes:
shape: (8, 2)
┌────────────────┬──────────┐
│ country        ┆ athletes │
│ ---            ┆ ---      │
│ str            ┆ u32      │
╞════════════════╪══════════╡
│ United States  ┆ 13748    │
│ United Kingdom ┆ 7565     │
│ Germany        ┆ 2138     │
│ Canada         ┆ 1144     │
│ France         ┆ 903      │
│ Japan          ┆ 857      │
│ Netherlands    ┆ 830      │
│ Brazil         ┆ 652      │
└────────────────┴──────────┘
